
# Boltz2-Notebook: Diffusion-Based Protein–Ligand Structure Prediction & Affinity Analysis


![Python](https://img.shields.io/badge/Python-3.10-blue?logo=python)
![CUDA](https://img.shields.io/badge/CUDA-Enabled-green?logo=nvidia)
![Boltz2](https://img.shields.io/badge/Model-Boltz2-purple)
![Platform](https://img.shields.io/badge/Platform-Colab%20|%20Linux-lightgrey?logo=googlecolab)
![License](https://img.shields.io/badge/License-MIT-orange)
![Status](https://img.shields.io/badge/Status-Active-success)
![Build](https://img.shields.io/badge/Build-Stable-brightgreen)
![Contributions](https://img.shields.io/badge/Contributions-Welcome-blue)
<br>

---

## Boltz2-Notebook Overview

**Boltz2-Notebook** is an **interactive Google Colaboratory platform** for **diffusion-based protein–ligand structure prediction** and **binding affinity estimation**.  
It integrates the **Boltz2 deep learning model** into a single, automated notebook environment — eliminating the need for local GPU setup, command-line execution, or YAML configuration.

Developed to enhance accessibility and usability, **Boltz2-Notebook** provides a fully guided workflow from setup to post-prediction analysis.  
It features a **graphical interface**, **3D molecular visualization**, and **automated confidence & affinity dashboards** — all within Google Colab.

---

###  Pipeline Overview
1. **Input**: Provide a protein sequence (and optional ligands).  
2. **YAML Generation**: The sequence is formatted into a YAML config.  
3. **MSA Search**: Boltz2 fetches multiple sequence alignments (MSA) using online servers.  
4. **Structure Prediction**: The neural network predicts 3D coordinates using diffusion and recycling steps.  
5. **Output**: Results include 3D models (CIF/PDB), confidence scores (**pLDDT**), and error heatmaps (**PAE**).  
6. **Visualization**: The notebook displays the predicted structure and confidence plots.  

---

 **Note:** This notebook automates the full Boltz2 workflow, from setup to visualization, with **color-coded status** and **interactive outputs**.  

---

##  Credits & Authorship

- **Notebook Developer:** Atharva Tilewale & Dr. Dhaval Patel
- **Affiliation:** Gujarat Biotechnology University | Bioinformatics & Computational Biology  
- **GitHub Repository:** [Boltz2-Notebook](https://github.com/AtharvaTilewale/boltz2-notebook)  
- **Contact:** [LinkedIn](https://www.linkedin.com/in/atharvatilewale) | [GitHub](https://github.com/AtharvaTilewale)  

**Acknowledgements:**  
- **Boltz2 framework**: [Original Boltz repository](https://github.com/jwohlwend/boltz) by J. Wohlwend and collaborators.  
- **Dependencies:** PyTorch, Biopython, NumPy, Matplotlib, Py3Dmol, PyYAML.  
- Special thanks to the **open-source community** for providing tools that make structural bioinformatics more accessible.  

---

## Cite
If you use this notebook, please **cite the following repository**:

[![GitHub Repo](https://img.shields.io/badge/GitHub-Boltz--Notebook-181717?logo=github)](https://github.com/AtharvaTilewale/Boltz-Notebook)

- Passaro, S., Corso, G., Wohlwend, J., Reveiz, M., Thaler, S., Somnath, V. R., Getz, N., Portnoi, T., Roy, J., Stark, H., Kwabi-Addo, D., Beaini, D., Jaakkola, T., & Barzilay, R. (2025).  
  **Boltz-2: Towards Accurate and Efficient Binding Affinity Prediction.** *bioRxiv.*  
    [![bioRxiv Boltz2](https://img.shields.io/badge/bioRxiv-Boltz2-red)](https://doi.org/10.1101/2025.06.14.659707)

- Wohlwend, J., Corso, G., Passaro, S., Getz, N., Reveiz, M., Leidal, K., Swiderski, W., Atkinson, L., Portnoi, T., Chinn, I., Silterra, J., Jaakkola, T., & Barzilay, R. (2024).  
  **Boltz-1: Democratizing Biomolecular Interaction Modeling.** *bioRxiv.*  
    [![bioRxiv Boltz1](https://img.shields.io/badge/bioRxiv-Boltz1-orange)](https://doi.org/10.1101/2024.11.19.624167)

- Mirdita, M., Schütze, K., Moriwaki, Y., Heo, L., Ovchinnikov, S., & Steinegger, M. (2022).  
  **ColabFold: Making protein folding accessible to all.** *Nature Methods.*  
    [![ColabFold](https://img.shields.io/badge/ColabFold-Reference-yellow)](https://doi.org/10.1038/s41592-022-01488-1)


---

In [7]:
# @title Install Dependencies and Boltz2 with CUDA support
import sys
import subprocess
import threading
import time
import os
import shutil
import torch

os.chdir("/content/")

# ANSI color codes for colored output
class Color:
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    RESET = "\033[0m"

# ---------------- GPU CHECK ----------------
print(f"{Color.CYAN}[i] Checking GPU availability...{Color.RESET}")
if not torch.cuda.is_available():
    print(f"{Color.RED}[✘] No GPU detected!{Color.RESET}")
    print(f"{Color.YELLOW}Please change runtime to 'GPU' (T4 or higher).{Color.RESET}")
    print(f"{Color.CYAN}Runtime > Change Runtime Type > Select any available GPU from Hardware Accelerator.{Color.RESET}")
    sys.exit(1)
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"{Color.GREEN}[✔] GPU detected:{Color.RESET} {gpu_name}")

# ---------------- INSTALL STEPS ----------------
repo_dirs = ["boltz2-notebook"]

steps = [
    {
        "loader": f"{Color.CYAN}Cloning Notebook Modules...{Color.RESET}",
        "done":   f"[{Color.GREEN}✔{Color.RESET}] Notebook modules cloned successfully.",
        "fail":   f"[{Color.RED}✘{Color.RESET}] Boltz-Notebook clone failed.",
        "cmd": ["git", "clone", "https://github.com/AtharvaTilewale/boltz2-notebook.git"]
    },
]

def loader(msg, stop_event):
    symbols = ["-", "\\", "|", "/"]
    i = 0
    while not stop_event.is_set():
        sys.stdout.write(f"\r[{symbols[i % len(symbols)]}] {msg}   ")
        sys.stdout.flush()
        time.sleep(0.1)
        i += 1
    sys.stdout.write("\r" + " " * (len(msg) + 10) + "\r")

# Step 1: Remove repo if it exists
for repo in repo_dirs:
    if os.path.isdir(repo):
        print(f"{Color.YELLOW}[i] Repository already exists. Removing '{repo}'...{Color.RESET}")
        try:
            shutil.rmtree(repo)
            print(f"[{Color.GREEN}✔{Color.RESET}] Existing repository '{repo}' removed.")
        except Exception as e:
            print(f"[{Color.RED}✘{Color.RESET}] Failed to remove '{repo}': {e}")
            raise

all_success = True

# Main steps
for step in steps:
    stop_event = threading.Event()
    t = threading.Thread(target=loader, args=(step["loader"], stop_event))
    t.start()
    try:
        subprocess.run(step["cmd"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        stop_event.set()
        t.join()
        print(step["done"])
    except Exception as e:
        stop_event.set()
        t.join()
        print(f"{step['fail']} {e}")
        all_success = False
        break

# Run setup if clone worked
if all_success:
    %run /content/boltz2-notebook/scripts/v1/setup.py
    print(f"{Color.GREEN}All steps completed successfully.{Color.RESET}")

[i] Checking GPU availability...
[✔] GPU detected: NVIDIA A100-SXM4-40GB
[✔] Notebook modules cloned successfully.
 ===Initialising Setup=== 


[i] Repository already exists. Removing 'boltz'...
[✔] Existing repository 'boltz' removed.
[✔] Boltz cloned successfully.
[✔] Dependencies installed successfully.
[✔] Validation complete.
All steps completed successfully.


In [8]:

# @title Download CCD Dataset and Test Boltz2
import sys
import threading
import time
import os

# ANSI color codes for colored output
class Color:
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    RESET = "\033[0m"

def loader(msg, stop_event):
    symbols = ["-", "\\", "|", "/"]
    i = 0
    while not stop_event.is_set():
        sys.stdout.write(f"\r[{symbols[i % len(symbols)]}] {msg}   ")
        sys.stdout.flush()
        time.sleep(0.1)
        i += 1
    sys.stdout.write("\r" + " " * (len(msg) + 10) + "\r")
    sys.stdout.flush()

# Step 1: Create data directory
os.makedirs("/content/boltz_data", exist_ok=True)
os.chdir("/content/boltz_data/")

# Step 2: Write YAML file
yaml_content = f"""\
version: 1
sequences:
    - protein:
        id: [A]
        sequence: MVTPE
    - ligand:
        id: [B]
        ccd: SAH
"""
with open("/content/boltz_data/test.yaml", "w") as f:
    f.write(yaml_content)

# Step 3: Run boltz predict (silent)
step_msg = f"{Color.YELLOW}Downloading CCD Dataset...{Color.RESET}"
stop_event = threading.Event()
t = threading.Thread(target=loader, args=(step_msg, stop_event))
t.start()
try:
    import subprocess
    subprocess.run(
        ["boltz", "predict", "test.yaml", "--use_msa_server"],
        cwd="/content/boltz_data",
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )
    stop_event.set()
    t.join()
    print(f"[{Color.GREEN}✔{Color.RESET}] CCD Dataset Downloaded and validated.")
except Exception as e:
    stop_event.set()
    t.join()
    print(f"[{Color.RED}✘{Color.RESET}] CCD Dataset Download or validation failed: {e}")


[✔] CCD Dataset Downloaded and validated.


In [9]:
*Cella disattivata: non più necessaria. Il calcolo è stato spostato sul notebook ufficiale.*

SyntaxError: invalid syntax (2523842384.py, line 1)

In [ ]:
*Cella disattivata: non più necessaria. Il calcolo è stato spostato sul notebook ufficiale.*

In [ ]:
*Cella disattivata: non più necessaria. Il calcolo è stato spostato sul notebook ufficiale.*

In [ ]:
*Cella disattivata: non più necessaria. Il calcolo è stato spostato sul notebook ufficiale.*

In [ ]:
*Cella disattivata: non più necessaria. Il calcolo è stato spostato sul notebook ufficiale.*

In [ ]:
# @title Estrai i dati di input da boltz2_inputs.zip
import zipfile
import os

zip_path = "/content/boltz2_inputs.zip"
extract_path = "/content/boltz_data"

# Assicuriamoci che la cartella di destinazione esista
os.makedirs(extract_path, exist_ok=True)

# Estrai il file zip se esiste
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"✅ File estratti con successo in {extract_path}!")
    print("Ora puoi eseguire la cella Step 6b.")
else:
    print(f"❌ Errore: il file {zip_path} non è stato trovato.")

In [10]:
import os
import glob
import subprocess

# @title Esecuzione Boltz-2 (Locale nel Notebook)
input_dir = "/content/boltz_data"
output_dir = "/content/boltz_results"
os.makedirs(output_dir, exist_ok=True)

# Trova tutti i file fasta o yaml estratti
input_files = glob.glob(f"{input_dir}/**/*.fasta", recursive=True) + glob.glob(f"{input_dir}/**/*.yaml", recursive=True)

if not input_files:
    print("❌ Nessun file .fasta o .yaml trovato in boltz_data. Assicurati che lo zip contenga i file corretti.")
else:
    print(f"✅ Trovati {len(input_files)} file. Avvio predizione Boltz-2...")

    for file_path in input_files:
        file_name = os.path.basename(file_path)
        print(f"\n⏳ Elaborazione di: {file_name}")
        try:
            # Esegue il comando boltz predict per ogni file, usando il server MSA pubblico
            result = subprocess.run(
                ["boltz", "predict", file_path, "--out_dir", output_dir, "--use_msa_server"],
                capture_output=True, text=True, check=True
            )
            print(f"✅ Predizione completata per {file_name}!")
        except subprocess.CalledProcessError as e:
            print(f"❌ Errore durante l'elaborazione di {file_name}:")
            print(e.stderr)

    print(f"\n🎉 Tutte le elaborazioni completate! I risultati sono in {output_dir}")


✅ Trovati 2 file. Avvio predizione Boltz-2...

⏳ Elaborazione di: test.yaml
✅ Predizione completata per test.yaml!

⏳ Elaborazione di: hparams.yaml
✅ Predizione completata per hparams.yaml!

🎉 Tutte le elaborazioni completate! I risultati sono in /content/boltz_results


In [ ]:
# @title Step 6b: Esporta i dati per il Notebook ufficiale di Boltz-2 (Fallback)
import shutil
import os
import glob
from google.colab import files

# Directory dove si trovano i file (modifica se i tuoi Top 50 sono in un'altra cartella)
source_dir = "/content/boltz_data"
export_dir = "/content/boltz2_export_files"
zip_filename = "/content/boltz2_export.zip"

# Pulisci la directory di export se esiste già
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)
os.makedirs(export_dir, exist_ok=True)

# Trova tutti i file YAML e FASTA
yaml_files = glob.glob(f"{source_dir}/**/*.yaml", recursive=True)
fasta_files = glob.glob(f"{source_dir}/**/*.fasta", recursive=True)
all_files = yaml_files + fasta_files

print(f"Trovati {len(all_files)} file pronti per l'export.")

# Copia i file nella cartella di export
for f in all_files:
    # Evita conflitti di nomi se ci sono file con lo stesso nome in sottocartelle diverse
    base_name = os.path.basename(f)
    dest_path = os.path.join(export_dir, base_name)
    counter = 1
    while os.path.exists(dest_path):
        name, ext = os.path.splitext(base_name)
        dest_path = os.path.join(export_dir, f"{name}_{counter}{ext}")
        counter += 1
    shutil.copy(f, dest_path)

# Crea l'archivio ZIP
if os.path.exists(zip_filename):
    os.remove(zip_filename)

shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', export_dir)
print(f"\nArchivio creato: {zip_filename}")
print("Download in corso...")

# Avvia il download
files.download(zip_filename)

In [12]:
import shutil
from google.colab import files
import os

# @title Scarica i Risultati di Boltz-2
results_dir = "/content/boltz_results"
zip_path = "/content/boltz_results.zip"

if os.path.exists(results_dir):
    print("📦 Creazione dell'archivio zip in corso...")
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', results_dir)

    print("⬇️ Avvio del download...")
    files.download(zip_path)
else:
    print("❌ Errore: la cartella dei risultati non esiste. Assicurati che l'esecuzione precedente sia andata a buon fine.")


📦 Creazione dell'archivio zip in corso...
⬇️ Avvio del download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import os
import zipfile
import glob
import subprocess

# @title Estrazione ed Esecuzione Boltz-2 sui tuoi Peptidi
zip_path = "/content/boltz2_inputs.zip"
input_dir = "/content/boltz_data_peptides"
output_dir = "/content/boltz_results_peptides"

os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# 1. Estrazione del file zip
print("📦 Estrazione dei file in corso...")
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(input_dir)
    print(f"✅ File estratti con successo in {input_dir}!")
else:
    print(f"❌ Errore: il file {zip_path} non è stato trovato. Assicurati di averlo caricato su Colab.")

# 2. Ricerca dei file estratti
input_files = glob.glob(f"{input_dir}/**/*.fasta", recursive=True)

if not input_files:
    print("⚠️ Nessun file .fasta trovato dopo l'estrazione.")
else:
    print(f"🔬 Trovati {len(input_files)} peptidi reali. Avvio predizione Boltz-2...")

    for file_path in input_files:
        file_name = os.path.basename(file_path)
        print(f"\n⏳ Elaborazione di: {file_name}")
        try:
            result = subprocess.run(
                ["boltz", "predict", file_path, "--out_dir", output_dir, "--use_msa_server"],
                capture_output=True, text=True, check=True
            )
            print(f"✅ Predizione completata per {file_name}!")
        except subprocess.CalledProcessError as e:
            print(f"❌ Errore durante l'elaborazione di {file_name}:")
            print(e.stderr)

    print(f"\n🎉 Tutte le elaborazioni completate! I risultati sono in {output_dir}")


📦 Estrazione dei file in corso...
❌ Errore: il file /content/boltz2_inputs.zip non è stato trovato. Assicurati di averlo caricato su Colab.
⚠️ Nessun file .fasta trovato dopo l'estrazione.


In [14]:
import shutil
from google.colab import files

# @title Scarica i nuovi risultati corretti
results_dir = "/content/boltz_results_peptides"
zip_path = "/content/boltz_results_peptides.zip"

if os.path.exists(results_dir):
    print("📦 Creazione dell'archivio zip dei tuoi peptidi in corso...")
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', results_dir)

    print("⬇️ Avvio del download...")
    files.download(zip_path)
else:
    print("❌ Nessun risultato da scaricare.")


📦 Creazione dell'archivio zip dei tuoi peptidi in corso...
⬇️ Avvio del download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>